# ATML PA4 — Task 1 (Zero-Manual-Copy Colab Pro Runner)

Open this notebook directly from GitHub in Colab, pick **one GPU tier** (recommended: **L4**; T4 also works) in `Runtime -> Change runtime type`, and click **Run all**. Nothing needs to be manually copied to Google Drive beforehand.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/umairs5/atml-pa4.git'
DATA_REPO_URL = 'https://github.com/AbDu11aHHH/atml-pa4-datasets.git'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/atml-pa4/results/task1'

In [ ]:
%cd /content
!if [ -d atml-pa4/.git ]; then git -C atml-pa4 pull --ff-only; else git clone {REPO_URL} atml-pa4; fi
%cd /content/atml-pa4
!if [ ! -d atml-pa4-datasets/.git ]; then git clone {DATA_REPO_URL} atml-pa4-datasets; fi
!cp configs/local_source_revisions.json atml-pa4-datasets/data/local_source_revisions.json
!nvidia-smi
!python -c "import torch; assert torch.cuda.is_available(), 'No CUDA GPU detected! Select L4 or T4 in Runtime -> Change runtime type.'"
!pip install -q -r requirements.txt

In [ ]:
# Materialize locked datasets on Colab's fast network (uses exact pinned commit SHAs) & run unit tests
!python atml-pa4-datasets/scripts/setup_data.py --public
!python atml-pa4-datasets/scripts/check_setup.py
!python -m unittest tests/test_task1.py -v

In [ ]:
# Run all Task 1 phases with automatic resumability (saves straight to Google Drive)
!python scripts/run_task1.py \
  --dataset-root ./atml-pa4-datasets \
  --output-dir {DRIVE_RESULTS_DIR} \
  --eval-batch-size 8 \
  --hs-batch-size 32 \
  --cal-batch-size 4

In [ ]:
# Compute paired 95% bootstrap CIs, summary CSV, plots, and qualitative cases
!python scripts/analyze_task1.py \
  --results-dir {DRIVE_RESULTS_DIR} \
  --dataset-root ./atml-pa4-datasets

import pandas as pd
df = pd.read_csv(f'{DRIVE_RESULTS_DIR}/summary_task1.csv')
display(df[['condition_id', 'wt2_ppl', 'c4_ppl', 'hs_n', 'hs_acc', 'hs_acc_norm', 'hs_512_acc_norm', 'achieved_sparsity', 'packed_mb', 'prefill_512_ms', 'decode_64_ms', 'peak_vram_mb']])

In [ ]:
# Archive lightweight summaries/plots/CIs to Drive + trigger browser download
!cd {DRIVE_RESULTS_DIR} && zip -rq /content/task1_results.zip summary_task1.csv bootstrap_ci_task1.json qualitative_hellaswag_cases.json env_and_accounting.json conditions/ dev_selection/ plots/
!cp /content/task1_results.zip {DRIVE_RESULTS_DIR}/task1_results.zip
from google.colab import files
files.download('/content/task1_results.zip')

In [ ]:
# Flush Google Drive writes and disconnect/terminate runtime to save Colab compute units
import time
from google.colab import drive, runtime

time.sleep(15)
drive.flush_and_unmount()
runtime.unassign()